In [1]:
!git clone https://github.com/CryAndRRich/chronolens.git

Cloning into 'chronolens'...
remote: Enumerating objects: 108, done.
remote: Counting objects: 100% (108/108), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 108 (delta 26), reused 99 (delta 20), pack-reused 0 (from 0)
Receiving objects: 100% (108/108), 15.35 MiB | 32.21 MiB/s, done.
Resolving deltas: 100% (26/26), done.


In [2]:
%cd /kaggle/working/chronolens
CHRONO_PATH = "/kaggle/working/chronolens"

/kaggle/working/chronolens


In [3]:
import sys

if CHRONO_PATH not in sys.path:
    sys.path.append(CHRONO_PATH)

In [4]:
import torch
import numpy as np
import pandas as pd
from torch.utils.data import Dataset, DataLoader

In [5]:
from config import *
from preprocess import *
from model import *
from utils import *

In [6]:
RANDOM_SEED = 42
set_seed(RANDOM_SEED)

data_generator = torch.Generator()
data_generator.manual_seed(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

Random seed set to 42
Device: cuda


In [7]:
INPUT_ROOT = "/kaggle/input/datasets/minhnguyn132/input-df26-v3"
WORK_DIR = "/kaggle/working"

X_TRAIN = f"{INPUT_ROOT}/X_train.csv"
X_VAL = f"{INPUT_ROOT}/X_val.csv"
X_TEST = f"{INPUT_ROOT}/X_test.csv"
Y_TRAIN = f"{INPUT_ROOT}/Y_train.csv"
Y_VAL = f"{INPUT_ROOT}/Y_val.csv"

SUBMISSION_FILE = f"{WORK_DIR}/submission.csv"
CHECKPOINT_FILE = f"{WORK_DIR}/best_model.pth"
RETRAIN_CHECKPOINT_FILE = f"{WORK_DIR}/best_model_retrain.pth"

#Có thể lựa chọn 1 trong 3: "chrono_r", "chrono_c" hoặc "chrono_g"
MODEL_NAME = "chrono_r"

In [8]:
data_manager = DataManager(
    input_root = INPUT_ROOT, 
    work_dir = WORK_DIR, 
    config_data = CONFIG_DATA,
    seed_worker = seed_worker,
    data_generator = data_generator,
    random_seed = RANDOM_SEED
)

In [9]:
x_train, y_train, x_val, y_val, x_test = data_manager.get_data()
train_loader, val_loader, test_loader = data_manager.get_dataloaders()
y_true = (y_val[CONFIG_DATA.ATTRIBUTE_COLS].values)[:, [1, 2, 3, 5, 6, 7]] 

In [10]:
best_epoch = train_model(
    model_name=MODEL_NAME,
    data_manager=data_manager,
    train_loader=train_loader, 
    val_loader=val_loader,  
    y_val=y_true, 
    num_epochs=1, # Đổi lại CONFIG_MODEL.NUM_EPOCHS để chạy full
    early_stopping=CONFIG_MODEL.EARLY_STOPPING,
    checkpoint_file=CHECKPOINT_FILE,
    device=DEVICE,
    verbose=True
)

Epoch 01/1 | LR: 0.000000 | Train loss: 1.2589 | Val score: 16.9332
  => Mô hình tốt nhất! (Val score: 16.9332)


In [11]:
model_kwargs = update_model_kwargs(
    data=data_manager,
    model_kwargs=CONFIG_MODEL.MODEL_KWARGS[MODEL_NAME]
)
model = get_model(
    name=MODEL_NAME,
    **model_kwargs  
).to(DEVICE)

model.load_state_dict(torch.load(CHECKPOINT_FILE, map_location=DEVICE))

model.eval()
val_pred = run_inference(model, val_loader, DEVICE)

In [12]:
get_stats(y_true, val_pred)


THUỘC TÍNH   | MAE        | MSE          | RMSE       | WMAPE     
----------------------------------------------------------------------------
Attr 1       | 1.4410     | 4.4950       | 2.1201     | 21.5206   
Attr 2       | 5.0019     | 53.6739      | 7.3262     | 39.2399   
Attr 3       | 24.8188    | 822.9973     | 28.6879    | 49.9427   
Attr 4       | 1.6388     | 5.6407       | 2.3750     | 24.4851   
Attr 5       | 4.8319     | 49.4395      | 7.0313     | 36.1877   
Attr 6       | 24.6744    | 819.2078     | 28.6218    | 49.5763   
MAE            : 10.4011
MSE            : 292.5757
RMSE           : 17.1048
WMAPE          : 44.9121
WMSE           : 16.9332


In [13]:
x_full = pd.concat([x_train, x_val], ignore_index=True)
y_full = pd.concat([y_train, y_val], ignore_index=True)

full_dataset = UserBehaviorDataset(
    x_df=x_full, 
    y_df=y_full, 
    feature_cols=data_manager.FEATURE_COLS,
    attr_cols=data_manager.ATTRIBUTE_COLS,
    augment=True
)
full_loader = DataLoader(
    full_dataset, 
    batch_size=1024, 
    shuffle=True, 
    num_workers=data_manager.NUM_WORKERS, 
    worker_init_fn=seed_worker, 
    generator=data_generator,
    pin_memory=True, 
    drop_last=True, 
    persistent_workers=True  
)

In [14]:
model = retrain_model(
    model_name=MODEL_NAME,
    data_manager=data_manager, 
    data_loader=full_loader,
    num_epochs=max(1, int(best_epoch * 1.1)),
    checkpoint_file=RETRAIN_CHECKPOINT_FILE,
    device=DEVICE,
    verbose=True
)

Retrain Epoch 01/1 | LR: 0.000000 | Train Loss: 1.1893


In [15]:
model.eval()
test_preds = run_inference(model, test_loader, DEVICE)
submission_df = pd.DataFrame({"id": x_test["id"]})

for idx, attr in enumerate(data_manager.ATTRIBUTE_LIST):
    submission_df[f"attr_{attr}"] = test_preds[:, idx].astype(np.uint16)
    
submission_df.to_csv(SUBMISSION_FILE, index=False)
print(submission_df)
print(submission_df.dtypes)

           id  attr_1  attr_2  attr_3  attr_4  attr_5  attr_6
0       n6r61       4      14      48       9      19      51
1       jkctd       8      18      47       7      16      50
2       a4060       3       6      51       2       5      52
3       fkbud      12      14      51       5      17      49
4       ui5gu       9      21      47       6      23      49
...       ...     ...     ...     ...     ...     ...     ...
108668  phk1w      11      17      49       5      14      51
108669  sgb1f       6      16      49       7      16      51
108670  e3s26       6      19      48       8      19      49
108671  d0kwv       5      16      48      10      21      51
108672  xfvdy       2      19      50      12      27      49

[108673 rows x 7 columns]
id        object
attr_1    uint16
attr_2    uint16
attr_3    uint16
attr_4    uint16
attr_5    uint16
attr_6    uint16
dtype: object
